# Chapter 12: Anti-Entropy and Dissemination

*From: Database Internals*

## TL;DR

- **Anti-entropy** is the cleanup crew for eventually consistent replication: primary delivery (coordinator → replicas) does best-effort propagation, and background mechanisms drag diverged replicas back into sync so convergence is bounded in time.
- Techniques split into **foreground** (piggybacked on client I/O — read repair, digest reads, hinted handoff) and **background** (run on schedules, walk the dataset — Merkle trees, bitmap version vectors).
- The four anti-entropy mechanisms optimize on two axes: **scope** (how much data do we compare — one key vs. whole dataset?) and **recency** (do we rediscover divergence from scratch or track it as it happens?). Merkle trees cover scope by hashing ranges; bitmap version vectors cover recency by logging dots.
- **Gossip protocols** push cluster metadata (membership, schemas, failures) epidemically — each node forwards to `f` random peers per round, reaching all N nodes in `O(log N)` rounds with built-in redundancy against failures.
- **Hybrid gossip** (Plumtree + HyParView) cuts redundancy by maintaining a spanning tree for eager push plus a lazy gossip fallback, and uses *small active + larger passive* partial views so nodes never have to maintain the full membership list.

## Problem Space

In an eventually consistent replicated database, **replica divergence is inevitable**. A write is sent to N replicas, but network blips, slow replicas, GC pauses, or short partitions mean some replicas will silently miss updates. Even under a quorum, the losing minority is allowed to lag. If we never repair that lag, divergence accumulates until reads become unpredictable.

The pragmatic design split is two-phase:

1. **Primary delivery** — the coordinator dispatches each write to the replica set and tries its best. Fast path, low redundancy.
2. **Anti-entropy** — background/foreground processes compare replica state and replay missing records. Slow path, guarantees convergence.

Orthogonally, *cluster metadata* (who is a member, which schema version is active, which nodes are suspected dead) is infrequent and small but must reach everyone **quickly and reliably**. A single broadcaster is a single point of failure and scales poorly. Gossip is the answer: every node helps spread the word, so the system tolerates the broadcaster crashing mid-announcement and keeps converging.

Keep both lenses in mind throughout the chapter: *data records* want the cheapest mechanism that eventually syncs, and *metadata* wants the fastest, most robust flood.

## Back-of-Envelope Estimation

Realistic numbers for why Merkle trees and gossip exist.

In [ ]:
# ----- 1. Naive pairwise replica comparison vs Merkle tree -----
records_per_replica = 1_000_000_000          # 1 B records
bytes_per_record = 100                       # small row
replica_bytes = records_per_replica * bytes_per_record
print(f"Replica dataset size: {replica_bytes / 1e9:,.0f} GB")

# To diff two replicas naively, we must ship at least one full replica.
naive_bytes_shipped = replica_bytes
print(f"Naive full-scan diff ships: {naive_bytes_shipped / 1e9:,.0f} GB per comparison")

# Merkle tree: group records into ranges of 10k, hash each leaf with SHA-256 (32 B).
range_size = 10_000
leaves = records_per_replica // range_size
hash_bytes = 32                              # SHA-256
# Perfect binary tree: total nodes ~= 2 * leaves - 1
tree_nodes = 2 * leaves - 1
tree_bytes = tree_nodes * hash_bytes
print(f"Merkle leaves: {leaves:,}  total nodes: {tree_nodes:,}")
print(f"Merkle tree on-wire size: {tree_bytes / 1e6:,.1f} MB (vs {replica_bytes / 1e9:,.0f} GB)")

# Suppose only 5 ranges actually differ. Walking top-down we touch ~log2(leaves)
# hashes per divergent leaf plus the leaf data.
import math
depth = math.ceil(math.log2(leaves))
divergent_ranges = 5
hashes_walked = divergent_ranges * depth
repair_data_bytes = divergent_ranges * range_size * bytes_per_record
print(f"Tree depth: {depth}  hashes walked for {divergent_ranges} divergent ranges: {hashes_walked}")
print(f"Data actually shipped for repair: {repair_data_bytes / 1e6:,.1f} MB")
print(f"Savings vs naive: {naive_bytes_shipped / repair_data_bytes:,.0f}x")

# ----- 2. Gossip convergence for metadata -----
N = 10_000        # cluster size
fanout = 3        # peers contacted per round
# Epidemic model: with fanout f, expected rounds to full coverage ~= log(N) / log(1+f)
rounds = math.ceil(math.log(N) / math.log(1 + fanout))
msgs_per_round = N * fanout
total_msgs = rounds * msgs_per_round
print(f"Gossip rounds to cover N={N:,} with fanout={fanout}: {rounds}")
print(f"Messages per round: {msgs_per_round:,}  total: {total_msgs:,}")
print(f"Redundancy factor (msgs per node delivered): {total_msgs / N:.1f}x")

The takeaways: a full-replica scan moves **100 GB per comparison**; the Merkle tree summary is ~**6 MB** and points us at exactly the handful of ranges that diverged. For 10k-node metadata dissemination, ~8 gossip rounds suffice — cheap, but each node pays a redundancy tax of ~25× messages, which is why hybrid gossip tries to trim it.

## High-Level Design

Foreground + background anti-entropy on a 3-replica data path:

```mermaid
graph LR
  Client --> Coord[Coordinator]
  Coord -- "write + read" --> R1[(Replica 1)]
  Coord -- "write + digest read" --> R2[(Replica 2)]
  Coord -- "write + digest read" --> R3[(Replica 3)]
  Coord -.->|"read repair (sync missing rows)"| R2
  Coord -.->|"hinted handoff (B is down)"| Hint[(Surrogate D)]
  Hint -.->|"replay on recovery"| R2
  R1 <-. "merkle sync (background)" .-> R2
  R2 <-. "merkle sync (background)" .-> R3
  R1 <-. "merkle sync (background)" .-> R3
```

Cluster metadata uses an entirely different plane — no coordinator, no replica set, just peers gossiping:

```mermaid
graph TD
  N1((N1)) --> N2((N2))
  N1 --> N5((N5))
  N2 --> N3((N3))
  N2 --> N6((N6))
  N5 --> N4((N4))
  N5 --> N7((N7))
  N3 --> N8((N8))
  N6 --> N9((N9))
  N7 --> N10((N10))
```

Each arrow is "node picked this peer at random this round and forwarded the hot update." Same message reaches everyone in ~log N rounds.

## Deep Dive

### 1. Read Repair and Digest Reads

Read repair is foreground anti-entropy that fires *during* a read. The coordinator fans out to the replica set, waits for responses, and if they disagree it ships the latest version back to the lagging replicas before (or after) returning to the client.

**Digest reads** are the optimization: instead of asking every replica for the full row, ask one replica for the full value and the rest for a hash of their value. If the hashes match, we're done. If any hash differs, upgrade to a full read on the mismatched replicas, merge, repair.

```mermaid
sequenceDiagram
  participant C as Client
  participant Co as Coordinator
  participant R1 as Replica 1
  participant R2 as Replica 2
  participant R3 as Replica 3
  C->>Co: GET key=k
  Co->>R1: full read(k)
  Co->>R2: digest read(k)
  Co->>R3: digest read(k)
  R1-->>Co: value v1, ts=10
  R2-->>Co: digest h(v1)
  R3-->>Co: digest h(v_old)
  Note over Co: h(v1) matches R2 but NOT R3 — R3 is lagging
  Co->>R3: full read(k)
  R3-->>Co: value v_old, ts=7
  Co->>R3: repair: write v1 ts=10
  Co-->>C: v1
```

**Trade-offs.** *Blocking* read repair guarantees read monotonicity under quorum (the client's next read sees at least what this one returned) but blocks on lagging replica acks, costing availability. *Asynchronous* read repair fires-and-forgets the repair task after the response, preserving latency but losing monotonicity. Also: read repair only fixes *what was queried*. Cold data never gets fixed this way — that's why we need the background mechanisms below.

### 2. Hinted Handoff

Write-side foreground anti-entropy. If a target replica is down, the coordinator (or a surrogate node stepping in under a sloppy quorum) writes a **hint** — a record tagged with "this was actually meant for B" — to its local hint log. When B is observed to be alive again, the hint is replayed and then deleted.

The canonical 5-node Dynamo-style sloppy quorum example:

```mermaid
graph LR
  Client --> A[A: coordinator]
  A -- replicate --> A
  A -- replicate --> C[C: replica]
  A -- "B is down; route to D with HINT for B" --> D[D: surrogate holds hint]
  B[B: failed] -. "recovers" .-> D
  D -- "forward hint, then delete" --> B
  A -. not used .- E[E]
```

**Trade-offs.** Sloppy quorum improves write availability during partitions: W=2 can still complete by hijacking D. But a subsequent read against `{B, C}` (if they are the ones partitioned away from A+D+E and receive the read) will miss that write until the hint is replayed — sloppy quorums trade consistency for availability. In Cassandra, hinted writes don't count toward the replication factor (unless `ANY` consistency is used) because hints aren't readable — they're only replay fuel.

### 3. Merkle Trees for Replica Reconciliation

Merkle trees give background anti-entropy the ability to locate divergent ranges in `O(log N)` hash comparisons instead of a pairwise row scan. Leaves hash *ranges* of records. Each interior node hashes the concatenation of its children. The root summarizes the entire dataset in 32 bytes.

Compare by walking: if two replicas' roots match, the dataset matches. If not, recurse into the subtree whose child hashes differ until you land on the specific range that needs reconciliation.

```mermaid
graph TD
  Root["Root = h(H12, H34)"]
  Root --> H12["H12 = h(H1, H2)"]
  Root --> H34["H34 = h(H3, H4)"]
  H12 --> H1["H1 = h(range 0..9999)"]
  H12 --> H2["H2 = h(range 10000..19999)"]
  H34 --> H3["H3 = h(range 20000..29999) DIVERGED"]
  H34 --> H4["H4 = h(range 30000..39999)"]
```

Walk: Root mismatches → H34 mismatches (H12 matches so prune that half) → H3 mismatches → compare rows in range `20000..29999`. Touched 5 hashes to localize the damage.

**Trade-offs.** Deeper trees (smaller ranges per leaf) give more precise repair but cost more to recompute on every write — a change in one record dirties the whole path from leaf to root. Shallower trees are cheaper to maintain but ship more row data once a divergence is found. Cassandra rebuilds Merkle trees on demand during repair rather than maintaining them live, trading build cost for write-path simplicity.

### 4. Bitmap Version Vectors

Where Merkle trees optimize **scope** (compare huge datasets cheaply), bitmap version vectors optimize **recency** (track divergence as it happens so you always know exactly what's missing).

Each write is a **dot** `(i, n)`: sequence number `i` assigned by coordinator `n`. Each node keeps a logical clock — a set of dots it has seen, either directly (coordinated by itself) or transitively (replicated from peers). Writes coordinated by node `n` on node `n` form a gap-free sequence; writes coordinated elsewhere may arrive out of order, producing gaps.

The clever compression: represent each peer's view as a pair `(prefix, bitmap)` where `prefix` is the highest contiguous sequence number seen for that peer, and `bitmap` encodes which *later* sequence numbers have also been seen (out-of-order).

Example state at P2's perspective — bitmap is interpreted left-to-right starting at `prefix + 1`:

| Peer | Dots seen (raw) | Compact form | Meaning |
|------|-----------------|--------------|---------|
| P1 | 1, 2, 3, 5, 6, 8 | `(3, 01101)` | consecutive through 3; of the next 5, positions 2, 3, 5 seen (i.e., 5, 6, 8) |
| P2 | 1, 2, 3, 4 | `(4, 0)` | P2 is authoritative for its own writes — no gaps |
| P3 | 1, 2 | `(2, 0)` | nothing newer from P3 yet |

Full clock at P2: `{P1: (3, 01101), P2: (4, 0), P3: (2, 0)}`.

During sync, P2 exchanges clocks with, say, P1. P1 sends `{P1: (8, 0), ...}` → P2 sees it is missing P1 dots `4` and `7`. P2 requests those specific records (stored in a **dotted causal container** that maps dots → values), patches its local state, and advances its prefix.

**Trade-offs.** Precise, causality-aware, O(peers) state per node once the prefix compacts. But if one peer stays offline forever, nobody can truncate the log of values that still need to reach it — log growth is unbounded in pathological scenarios.

### 5. Gossip Dissemination

Epidemic broadcast. Model each node's relationship to a given update with the SIR state machine:

```mermaid
flowchart LR
  S[Susceptible] -->|receives update| I[Infective]
  I -->|forwards to f random peers each round| I
  I -->|loses interest: too many duplicates seen| R[Removed]
```

Each round, every infective node picks `f` peers at random and pushes the update. Convergence in `O(log N)` rounds. Tuning knobs: **fanout `f`** (higher = faster + more redundant), **loss-of-interest policy** (probabilistic stop vs. threshold on duplicate count).

Simulation of infection spread (Python 3.11+, stdlib only, seeded):

In [ ]:
import random

def simulate_gossip(n=100, fanout=3, seed=42):
    random.seed(seed)
    nodes = list(range(n))
    infected = {0}  # node 0 starts with the update
    rounds = 0
    history = [len(infected)]
    while len(infected) < n:
        rounds += 1
        new_infections = set()
        # All currently-infected nodes each push to `fanout` random peers
        for src in list(infected):
            peers = random.sample(nodes, fanout)
            for p in peers:
                if p not in infected:
                    new_infections.add(p)
        infected |= new_infections
        history.append(len(infected))
        if rounds > 50:
            break  # safety
    return rounds, history

rounds, history = simulate_gossip(n=100, fanout=3)
print(f"Full coverage in {rounds} rounds")
print(f"Coverage per round: {history}")

# Sanity check: fanout=1 is noticeably slower and may take many more rounds.
rounds2, _ = simulate_gossip(n=100, fanout=1, seed=42)
print(f"With fanout=1: {rounds2} rounds")

**Trade-offs.** Gossip is wildly robust — partitions just slow it down, they don't stop it. Cost is message redundancy: each node receives the same update from multiple peers. For cluster metadata that's a cheap tax to pay; for bulk data it is not.

### 6. Hybrid Gossip and Partial Views

Pure gossip is robust but wasteful. Pure spanning-tree broadcast is optimal but brittle — a single broken edge isolates a subtree. **Plumtree** combines them: maintain an overlay spanning tree and push the full message eagerly over tree edges, but also lazy-push *message IDs* (not bodies) to off-tree peers over a gossip backplane. A node that hears an ID it hasn't seen pulls the body from the sender — this both delivers missed messages and repairs the tree.

```mermaid
graph TD
  A((A)) ==> B((B))
  A ==> C((C))
  B ==> D((D))
  B ==> E((E))
  C ==> F((F))
  C ==> G((G))
  A -.-> F
  B -.-> C
  D -.-> E
  E -.-> G
  F -.-> G
```

Solid (`==>`) = eager push along the spanning tree. Dotted (`-.->`) = lazy gossip carrying only message IDs for repair.

**HyParView** supplies the peer sampling under Plumtree. Each node maintains two views:

- **Active view** (small, e.g., ~5 peers) — the nodes it currently has open TCP connections to and uses for eager push. These are the spanning-tree neighbors.
- **Passive view** (larger, e.g., ~30) — a cheap cache of candidate peers used to replace active-view members that fail. Shuffled periodically with peers.

When a node suspects an active-view peer `P2` has died, it evicts `P2` and promotes a passive-view peer `P3` into the active view. The system never needs a global membership list: a new node just needs one seed to bootstrap into someone's active view.

**Trade-offs.** Eager push minimizes message count in steady state; lazy push guarantees eventual delivery under failure. Small active views mean fast tree but slower recovery if a peer dies (fewer fallbacks); larger passive views speed recovery at the cost of more shuffle traffic. HyParView prioritizes bootstrapping nodes to converge quickly to a stable overlay.

## Trade-offs and Alternatives

| Mechanism | Scope | Recency | Cost | Failure Mode |
|-----------|-------|---------|------|--------------|
| Read repair (+ digest) | Just-queried key | Current request only | Cheap on happy path (hash compare); upgrades to full read on mismatch | Cold data never repaired; async variant loses read monotonicity |
| Hinted handoff | Individual missed writes | Captured at write time | Storage on surrogate until replay; bounded by hint TTL | Sloppy quorum reads can miss not-yet-replayed hints; hints can be lost if surrogate crashes before replay |
| Merkle trees | Entire dataset | Rediscovered on each sync run | O(log N) hashes compared; recompute subtree on every write (or rebuild lazily) | Tree precision vs. recompute cost; two replicas with many divergences may still ship big ranges |
| Bitmap version vectors | All writes, tracked per-peer | Precise — knows exactly which dots are missing | O(peers) state; log growth if a peer stays offline | Log cannot be truncated until every peer has advanced the prefix; clock size grows with cluster size |
| Plain gossip | Cluster metadata (small) | Epidemic — best-effort eventual | O(log N) rounds, f × N msgs per round | Redundant messages; probabilistic convergence |
| Hybrid gossip (Plumtree + HyParView) | Cluster metadata | Eager push in steady state, lazy on failure | Much lower redundancy in steady state | Slower convergence right after topology churn; extra view-maintenance traffic |

## Key Takeaways

1. **Two-phase thinking.** Primary delivery is optimistic and fast; anti-entropy is the backstop that bounds convergence time. Every eventually consistent system needs both.
2. **Foreground mechanisms piggyback on I/O.** Read repair (read path) and hinted handoff (write path) are cheap because they share work with client requests — they're not real background jobs.
3. **Background mechanisms pick a specialization.** Merkle trees optimize *scope* (cheap full-dataset diffing); bitmap version vectors optimize *recency* (precise per-dot tracking). Neither dominates; systems often run both.
4. **Digest reads are the "free" win.** If hashes match, we avoided shipping a row. If they don't, we degrade to a full read. The happy path is almost always the happy path.
5. **Sloppy quorums trade consistency for availability at write time.** Dynamo/Riak will accept writes onto surrogates to keep W satisfied during failures, paying for it with possibly-stale reads until hints replay.
6. **Gossip reaches everyone in O(log N) rounds.** The cost is redundancy — every node receives duplicates. Worth it for cluster-wide metadata, wasteful for bulk data.
7. **Hybrid gossip (Plumtree + HyParView) is the practical sweet spot.** Eager push on a small spanning tree for efficiency, lazy push for repair, and partial views so nodes never need a global membership list. Used in Riak Core and Partisan.

## See Also

- [[01-read-repair-and-digest-reads]]
- [[02-hinted-handoff]]
- [[03-merkle-trees]]
- [[04-bitmap-version-vectors]]
- [[05-gossip-dissemination]]
- [[06-hybrid-gossip-and-partial-views]]